In [1]:
from dataclasses import dataclass
import os
from pathlib import Path
import zipfile
import requests
from typing import Optional, List, Dict, Any

In [2]:
CONFIG_FILE_PATH = Path("config/config.yaml")
PARAMS_FILE_PATH = Path("params.yaml")
GLOBAL_PATH = Path("/Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer")

In [3]:
import os
from typing import Any, Dict, Optional
from box.exceptions import BoxValueError
from ContentSummarizer.utils.logger import logger

from ensure import ensure_annotations
from box import ConfigBox
from pathlib import Path
import yaml



@ensure_annotations
def read_yaml(path_to_yaml: Path) -> ConfigBox:
    """
    Reads a YAML file and returns its contents as a ConfigBox object.

    Args:
        path_to_yaml (Path): Path to YAML file

    Returns:
        ConfigBox: Parsed YAML content
    """

    try:

        resolved_path = Path(GLOBAL_PATH, path_to_yaml).resolve()

        logger.info(
            "Reading YAML file: %s",
            resolved_path
        )

        logger.info(
            "Current Working Directory: %s",
            Path.cwd()
        )

        if not resolved_path.exists():

            logger.error(
                "YAML file does not exist: %s",
                resolved_path
            )

            raise FileNotFoundError(
                f"YAML file not found: {resolved_path}"
            )

        with open(
            resolved_path,
            "r",
            encoding="utf-8"
        ) as yaml_file:

            yaml_content = yaml.safe_load(yaml_file)

        logger.info(
            "Successfully loaded YAML file: %s",
            resolved_path
        )

        return ConfigBox(yaml_content)

    except Exception as e:

        logger.exception(
            "Error reading YAML file: %s",
            path_to_yaml
        )

        raise BoxValueError(
            f"Error reading YAML file at {path_to_yaml}: {e}"
        )
    
def create_directories(path_to_directories: list) -> None:

    try:

        logger.info(
            "Current Working Directory: %s",
            Path.cwd()
        )

        for path in path_to_directories:

            resolved_path = Path(GLOBAL_PATH, path).resolve()

            logger.info(
                "Original Path: %s | Resolved Path: %s",
                path,
                resolved_path
            )

            if resolved_path.is_file():

                logger.warning(
                    "Path %s is a file. Skipping.",
                    resolved_path
                )

                continue

            resolved_path.mkdir(
                parents=True,
                exist_ok=True
            )

            logger.info(
                "Directory created at: %s",
                resolved_path
            )

    except Exception as e:

        logger.exception(
            "Error creating directories: %s",
            str(e)
        )

        raise BoxValueError(
            f"Error creating directories: {e}"
        )
    
@ensure_annotations
def get_size(path: Path) -> str:
    """
    Gets the size of a file or directory in a human-readable format.

    Args:
        path (Path): The path to the file or directory.
    Returns:
        str: The size of the file or directory in a human-readable format.
    """
    try:
        if Path(GLOBAL_PATH, path).is_file():
            size = Path(GLOBAL_PATH, path).stat().st_size
        else:
            size = sum(Path(GLOBAL_PATH, f).stat().st_size for f in Path(GLOBAL_PATH, path).glob('**/*') if Path(GLOBAL_PATH, f).is_file())
        
        for unit in ['B', 'KB', 'MB', 'GB', 'TB']:
            if size < 1024:
                return f"{size:.2f} {unit}"
            size /= 1024
        return f"{size:.2f} PB"
    except Exception as e:
        logger.error(f"Error getting size for {path}: {e}")
        raise BoxValueError(f"Error getting size for {path}: {e}")

[2026-06-03 11:18:52] [INFO] [content_summarizer] [logger.py:102] Logger initialized successfully


In [4]:
@dataclass
class DataValidationConfig:
    root_dir: Path
    artifacts_dir: Path
    status_file: Path
    required_files: List[str]

In [ ]:
class ConfigurationManager:
    config_file_path: Path
    params_file_path: Path

    def __init__(self, config_file_path: Path = Path(GLOBAL_PATH, CONFIG_FILE_PATH), params_file_path: Path = Path(GLOBAL_PATH, PARAMS_FILE_PATH)):
        self.config_file_path = Path(GLOBAL_PATH, config_file_path)
        self.params_file_path = Path(GLOBAL_PATH, params_file_path)
        self.config = read_yaml(self.config_file_path)
        self.params = read_yaml(self.params_file_path)
        create_directories([self.config["artifacts_root"]])
    
    def get_data_validation_config(self) -> DataValidationConfig:
        config = self.config.data_validation
        create_directories([config.root_dir])
        data_validation_config = DataValidationConfig(
            root_dir=Path(config.root_dir),
            status_file=Path(config.status_file),
            artifacts_dir=Path(config.artifacts_dir),
            required_files=config.required_files
        )
        return data_validation_config

In [6]:
class DataValidation:

    """ 
    This class is responsible for validating the data. It checks if the required files are present in the root directory and updates the status file accordingly.
    """
    def __init__(self, config: DataValidationConfig):
        self.config = config
        create_directories([self.config.root_dir])

    def validate(self) -> bool:
        """
        Validates the data by checking if the required files are present in the root directory and updates the status file accordingly.

        Returns:
            bool: True if the data is valid, False otherwise.
        """
        try:
            logger.info("Starting data validation.")
            logger.info(f"Checking for required files in {GLOBAL_PATH, self.config.artifacts_dir}")
            path = Path(GLOBAL_PATH, self.config.artifacts_dir).resolve()
            logger.info(f"Validation directory: {path}")

            if not path.exists():
                logger.error(f"Directory does not exist: {path}")
                return False

            actual_files = [file.name for file in path.rglob("*")]
            logger.info(f"Found files: {actual_files}")
            missing_files = [file for file in self.config.required_files if file not in actual_files]
            logger.info(f"Missing files: {missing_files}")
            if missing_files:
                logger.warning(f"Missing files: {missing_files}")
                with open(Path(GLOBAL_PATH, self.config.status_file), "w") as f:
                    f.write(f"Data validation failed. Missing files: {missing_files}")
                return False
            else:
                logger.info("All required files are present.")
                with open(Path(GLOBAL_PATH, self.config.status_file), "w") as f:
                    f.write("Data validation successful. All required files are present.")
                return True
        except Exception as e:
            logger.error(f"Error during data validation: {e}")
            with open(Path(GLOBAL_PATH, self.config.status_file), "w") as f:
                f.write(f"Data validation failed due to error: {e}")
            return False
    

In [7]:
config = ConfigurationManager(Path(CONFIG_FILE_PATH), Path(PARAMS_FILE_PATH))
dataValidationConfig = config.get_data_validation_config()
print(dataValidationConfig)

[2026-06-03 11:18:53] [INFO] [content_summarizer] [2511429230.py:29] Reading YAML file: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/config/config.yaml
[2026-06-03 11:18:53] [INFO] [content_summarizer] [2511429230.py:34] Current Working Directory: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/research
[2026-06-03 11:18:53] [INFO] [content_summarizer] [2511429230.py:58] Successfully loaded YAML file: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/config/config.yaml
[2026-06-03 11:18:53] [INFO] [content_summarizer] [2511429230.py:29] Reading YAML file: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/params.yaml
[2026-06-03 11:18:53] [INFO] [content_summarizer] [2511429230.py:34] Current Working Directory: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/research
[2026-06-

In [8]:
dataValidation = DataValidation(dataValidationConfig)
is_valid = dataValidation.validate()
print(f"Is the data valid? {is_valid}")

[2026-06-03 11:18:53] [INFO] [content_summarizer] [2511429230.py:80] Current Working Directory: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/research
[2026-06-03 11:18:53] [INFO] [content_summarizer] [2511429230.py:89] Original Path: artifacts/data_validation | Resolved Path: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/artifacts/data_validation
[2026-06-03 11:18:53] [INFO] [content_summarizer] [2511429230.py:109] Directory created at: /Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer/artifacts/data_validation
[2026-06-03 11:18:53] [INFO] [content_summarizer] [496559245.py:18] Starting data validation.
[2026-06-03 11:18:53] [INFO] [content_summarizer] [496559245.py:19] Checking for required files in (PosixPath('/Users/gtanuj7987/Documents/Data Science Naik/Content Summarize/project-content-summarizer'), PosixPath('artifacts/data_ingestion'))
[2026-06-03 1